In [29]:
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(host=host, port=port)

Create schema.

In [30]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768

id_field = FieldSchema(
    name="id", dtype=DataType.INT64, is_primary=True, description="Primary id"
)
text = FieldSchema(
    name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text"
)
embedding_text = FieldSchema(
    "embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=VECTOR_LENGTH,
    description="Embedded text",
)

fields = [id_field, text, embedding_text]

schema = CollectionSchema(
    fields=fields,
    auto_id=True,
    enable_dynamic_field=True,
    description="RAG Texts collection",
)

Create a collection with the given schema.

In [31]:
COLLECTION_NAME = "rag_texts_and_embeddings"

if milvus_client.has_collection(COLLECTION_NAME):
    milvus_client.drop_collection(COLLECTION_NAME)

milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64},  # lower values for speed
)

milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

print(milvus_client.list_collections())

print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 470186346743760319, 'consistency_level': 2, 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'created_timestamp': 470186945862696978, 'update_timestamp': 470186945862696978}


In [32]:
# define data source and destination
## the document origin destination from which document will be downloaded
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [33]:
import os
import requests


def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)


download_pdf_data(pdf_url, file_name)

In [34]:
import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)


extract_pdf_text(file_name, file_json)

Vectorize data.

In [35]:
import torch
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append(
            {"page_num": page_num, "embedding": embeddings[page_num].tolist()}
        )

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)


model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

In [36]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with (
        open(os.path.join(data_dir, file_json), "r") as t_f,
        open(os.path.join(data_dir, embeddings_json), "r") as e_f,
    ):
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data = list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))

        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text": text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

milvus_client.load_collection("rag_texts_and_embeddings")

In [37]:
def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings",
        data=[embedded_query],
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"],
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [38]:
result

data: [[{'id': 470186346742553370, 'distance': 29.12517547607422, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mogł

In [ ]:
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-2.5-flash"


def generate_response(prompt: str):
    try:
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [47]:
def build_prompt(context: str, query: str, instructions: str = "") -> str:
    prompt = f"""
Odpowiedz na pytanie, korzystając WYŁĄCZNIE z poniższego kontekstu.
Jeśli kontekst nie zawiera odpowiedzi, napisz: "Nie znalazłem tej informacji w źródłach."
Nie zmyślaj i nie korzystaj z wiedzy spoza kontekstu.
{instructions}

Kontekst:
\"\"\"
{context}
\"\"\"

Pytanie: {query}

Odpowiedź:
"""
    return prompt


def rag(model, query: str, instructions: str = "") -> str:
    result = search(model, query)
    hits = result[0]
    if not hits:
        return "Content not found in Milvus."

    context = "\n\n".join(hit["entity"]["text"] for hit in hits)
    prompt = build_prompt(context, query, instructions)

    return generate_response(prompt)

Test the basic RAG with a few queries.

In [48]:
queries = [
    "Czym jest sztuczna inteligencja?",
    "Kiedy powstał termin sztuczna inteligencja?",
    "Jak działa uczenie maszynowe?",
]

for q in queries:
    print("Q:", q)
    print("A:", rag(model, q))
    print("-" * 40)

Q: Czym jest sztuczna inteligencja?
A: Sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań, które wymagają ludzkiej inteligencji. Te zadania obejmują rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się, planowanie i wiele innych. Sztuczna inteligencja byłaby czymś (programem, maszyną) symulującym inteligencję naturalną, ludzką, a jej głównym celem jest stworzenie systemów, które są zdolne do myślenia i podejmowania decyzji na sposób przypominający ludzki. W języku potocznym "sztuczny" oznacza to, co jest wytworem mającym naśladować coś naturalnego.
----------------------------------------
Q: Kiedy powstał termin sztuczna inteligencja?
A: Nie znalazłem tej informacji w źródłach.
----------------------------------------
Q: Jak działa uczenie maszynowe?
A: Uczenie maszynowe to proces, który systemy komputerowe wykonują, aby osiągnąć sztuczną inteligencję. Wykorzystuje algo

Experiments with the prompt. Each one only changes the extra instructions, the grounding rules stay the same.

In [49]:
query = "Czym jest sztuczna inteligencja?"

experiments = {
    "style: pirate": "Odpowiadaj jak pirat, używaj piracich zwrotów.",
    "language: English": "Answer in English, even though the context is in Polish.",
    "format: 3 bullet points": "Odpowiedz dokładnie w trzech punktach (bullet points), bez żadnego dodatkowego tekstu.",
    "persona: grumpy professor": "Odpowiadaj jak zrzędliwy profesor, który jest zirytowany takim pytaniem.",
    "constraint: max 30 words": "Odpowiedz maksymalnie w 30 słowach.",
}

for name, instructions in experiments.items():
    print(f"=== {name} ===")
    print(rag(model, query, instructions))
    print()

=== style: pirate ===
Ahoj, kamracie! Z przyjemnością ci powiem, co to jest ta sztuczna inteligencja, wedle mapy, którą mi dali!

Sztuczna inteligencja, zwana też AI, to nic innego jak obszar informatyki, co to się skupia na stworzeniu programów komputerowych zdolnych do wykonania zadań, które wymagają bystrej, ludzkiej inteligencji. To takie coś, co symuluje naturalną, ludzką inteligencję, mości panie! Jej głównym celem jest stworzenie systemów, co to potrafią myśleć i podejmować decyzje niczym człowiek. Czasem nawet przewyższają naturalne funkcje, na przykład w liczeniu czy pamięci, unikając pomyłek!

=== language: English ===
Artificial intelligence (AI) is an area of computer science that focuses on creating computer programs capable of performing tasks that require human intelligence. These tasks include pattern recognition, natural language understanding, decision-making, learning, planning, and many others. AI would be something (a program, a machine) simulating natural, human i

Hallucination probe. Questions that are not in the document.

In [50]:
off_topic_queries = [
    "Jaka jest stolica Australii?",
    "Ile kosztuje GPT-4 w API?",
    "Kto wygrał Mistrzostwa Świata w piłce nożnej w 2018?",
]

for q in off_topic_queries:
    print("Q:", q)
    print("A:", rag(model, q))
    print("-" * 40)

Q: Jaka jest stolica Australii?
Error generating response: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash, description: Generate content requests per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 46.383365082s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests p

Do the same probe, but together with a style change, to see if the model stays grounded.

In [51]:
for q in off_topic_queries:
    print("Q:", q)
    print("A:", rag(model, q, "Odpowiadaj jak pirat, używaj piracich zwrotów."))
    print("-" * 40)

Q: Jaka jest stolica Australii?
Error generating response: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash, description: Generate content requests per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 45.764856578s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests p

Limit darmowych zapytań jest dość mały, więc nie mogę wykonać wszystkich poleceń. Z tego co widać, na początkowych zapytaniach, model wraz z RAGiem dość dobrze sobie radzi i odpowiada na pytania poprawnie, nawet z dodatkowymi wymaganiami w `query`.